In [7]:
!pip install -q openai python-dotenv tqdm

from google.colab import drive
drive.mount('/content/drive')

import os
import time
from getpass import getpass
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI


Mounted at /content/drive


In [8]:
# Path file tafsir yang akan dipakai
PATH_TAFSIR = "/content/drive/MyDrive/fp_quran/tafsir_clean.csv"
PATH_TAFSIR = Path(PATH_TAFSIR)

print("File tafsir ada?", PATH_TAFSIR.exists())
df_tafsir = pd.read_csv(PATH_TAFSIR)
print("Shape:", df_tafsir.shape)
print("Kolom:", df_tafsir.columns.tolist())
df_tafsir.head()


File tafsir ada? True
Shape: (6236, 6)
Kolom: ['surah', 'ayah', 'arabic_text', 'indonesian_translation', 'tafsir', 'tafsir_id']


,surah,ayah,arabic_text,indonesian_translation,tafsir,tafsir_id
0,Al-Fātiḥah,1,بِسْمِ اللّٰهِ الرَّحْمٰنِ الرَّحِيْمِ,Dengan nama Allah Yang Maha Pengasih lagi Maha...,Aku memulai bacaan Al-Qur'an dengan menyebut n...,Al-Fātiḥah :1
1,Al-Fātiḥah,2,اَلْحَمْدُ لِلّٰهِ رَبِّ الْعٰلَمِيْنَۙ,"Segala puji bagi Allah, Tuhan1) semesta alam",Segala puji kita persembahkan hanya untuk Alla...,Al-Fātiḥah :2
2,Al-Fātiḥah,3,الرَّحْمٰنِ الرَّحِيْمِۙ,"Yang Maha Pengasih lagi Maha Penyayang,","Dialah Yang Maha Pengasih, Pemilik dan sumber ...",Al-Fātiḥah :3
3,Al-Fātiḥah,4,مٰلِكِ يَوْمِ الدِّيْنِۗ,Pemilik hari Pembalasan.2),Dialah satu-satunya Pemilik hari Pembalasan da...,Al-Fātiḥah :4
4,Al-Fātiḥah,5,اِيَّاكَ نَعْبُدُ وَاِيَّاكَ نَسْتَعِيْنُۗ,Hanya kepada Engkaulah kami menyembah dan hany...,"Atas dasar itu semua, hanya kepada Engkaulah k...",Al-Fātiḥah :5


In [9]:
COL_SURAH     = "surah"
COL_AYAH      = "ayah"
COL_TAFSIR_ID = "tafsir_id"
COL_TAFSIR    = "tafsir"

# Buat kolom ID numerik kalau belum ada
if "id" not in df_tafsir.columns:
    df_tafsir["id"] = range(1, len(df_tafsir) + 1)

COL_ID = "id"

df_tafsir[[COL_ID, COL_SURAH, COL_AYAH, COL_TAFSIR_ID, COL_TAFSIR]].head()


,id,surah,ayah,tafsir_id,tafsir
0,1,Al-Fātiḥah,1,Al-Fātiḥah :1,Aku memulai bacaan Al-Qur'an dengan menyebut n...
1,2,Al-Fātiḥah,2,Al-Fātiḥah :2,Segala puji kita persembahkan hanya untuk Alla...
2,3,Al-Fātiḥah,3,Al-Fātiḥah :3,"Dialah Yang Maha Pengasih, Pemilik dan sumber ..."
3,4,Al-Fātiḥah,4,Al-Fātiḥah :4,Dialah satu-satunya Pemilik hari Pembalasan da...
4,5,Al-Fātiḥah,5,Al-Fātiḥah :5,"Atas dasar itu semua, hanya kepada Engkaulah k..."


In [11]:
PATH_INDO_QA = "/content/drive/MyDrive/fp_quran/indo_islamic_qa_raw.csv"  # file 27 query tadi

if os.path.exists(PATH_INDO_QA):
    df_indo_qa = pd.read_csv(PATH_INDO_QA)
    ANCHOR_QUERIES = (
        df_indo_qa["query"]
        .dropna()
        .astype(str)
        .sample(n=min(6, len(df_indo_qa)), random_state=42)
        .tolist()
    )
else:
    ANCHOR_QUERIES = [
        "doa nabi musa",
        "saya sedang sakit bolehkah saya tidak puasa?",
        "dosa saya sudah banyak apakah saya masih bisa diampuni Allah?",
        "ayat yang bilang semua kesulitan pasti ada jalan keluar",
        "apa hukumnya musik dalam islam?",
        "bagaimana cara memuliakan orang tua menurut Al-Qur'an?",
    ]

print("Contoh anchor queries:")
for q in ANCHOR_QUERIES:
    print("-", q)


Contoh anchor queries:
- gimana caranya ningkatin iman?
- ayat yang memerintahkan kita untuk sholat
- Bagaimana cara mendapatkan ketenangan hati melalui ajaran Al-Quran?
- Bagaimana jika kita kesulitan dalam membayar hutang?
- doa nabi musa
- Apakah tugas manusia hanya perlu hidup semaksimal nya dan ikhlas ketika diberi cobaan?


In [10]:
DEEPSEEK_API_KEY = getpass("Masukkan DEEPSEEK_API_KEY: ")

client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com",
)

# Tes singkat
resp = client.chat.completions.create(
    model="deepseek-chat",
    messages=[{"role": "user", "content": "Halo, ini test dari FP Qur'an IR."}],
    max_tokens=20,
)
print("Respon test:", resp.choices[0].message.content)


Masukkan DEEPSEEK_API_KEY: ··········
Respon test: Halo! Selamat datang di FP Qur'an IR test. 

Saya siap membantu Anda


In [19]:
N_QUERY_MIN = 3
N_QUERY_MAX = 8  # target 3–8 query per ayat

BANNED_PHRASES = [
    "tafsir ini",
    "tafsir di atas",
    "teks di atas",
    "penjelasan di atas",
    "ayat ini",
    "ayat tersebut",
    "menurut tafsir",
    "dalam teks tersebut",
    "dalam penjelasan ini",
]


def build_prompt_for_tafsir(tafsir_text: str) -> str:
    # batasi panjang tafsir agar hemat token tanpa menghilangkan inti makna
    tafsir_short = str(tafsir_text)[:1500]

    contoh_str = "\n".join(f"- {q}" for q in ANCHOR_QUERIES)

    prompt = f"""
Teks berikut adalah penjelasan (tafsir) tentang satu ayat Al-Qur'an:

\"\"\"{tafsir_short}\"\"\"

Bayangkan ada seorang muslim awam yang sedang mencari jawaban di internet.
Ia TIDAK membaca teks di atas, tetapi masalah yang ia alami sama dengan isi tafsir tersebut.

TUGAS:
- Buat antara {N_QUERY_MIN} sampai {N_QUERY_MAX} kalimat pencarian (query) dalam Bahasa Indonesia.
- Gaya bahasa natural seperti orang bertanya di Google atau aplikasi Islami.
- Panjang tiap query idealnya 5–20 kata (boleh sedikit lebih panjang jika memang perlu, maksimal sekitar 25 kata).
- Gunakan variasi bentuk:
  - pertanyaan lengkap: "bagaimana ...", "mengapa ...", "apa ...",
  - frasa pencarian: "ayat tentang ...", "dalil tentang ...", "doa ...", "ancaman bagi ...".
- Usahakan setidaknya dua query berbentuk frasa seperti:
  - "ayat tentang ...", "dalil tentang ...", atau "doa ...".
- Fokus pada aspek yang PALING KHAS dari tafsir ini
  (misalnya: sifat munafik, larangan riba, keutamaan sabar),
  bukan tema umum yang bisa berlaku di banyak ayat
  (misalnya: "ingin bahagia dunia akhirat" yang terlalu luas).

PENTING:
- JANGAN menyebut "tafsir ini", "teks di atas", "penjelasan di atas", "ayat ini", atau frase sejenis.
- JANGAN menyebut bahwa ini dataset, penelitian, model AI, atau tugas akhir.
- JANGAN menyalin kalimat tafsir secara utuh; ubah menjadi gaya pencarian pengguna.
- Hindari menyebut nomor surah/ayat secara eksplisit kecuali benar-benar perlu.

Contoh gaya query (JANGAN disalin, hanya gayanya saja):
{contoh_str}

Format keluaran:
Tuliskan satu query per baris, diberi nomor seperti:
1. <query pertama>
2. <query kedua>
3. <query ketiga>
dst.
"""
    return prompt



def parse_queries_from_llm_output(text: str):
    lines = text.splitlines()
    queries = []

    for line in lines:
        line = line.strip()
        if not line:
            continue
        # buang prefix nomor "1. ", "2) ", "- " dst
        line = line.lstrip("0123456789.:-) ")
        q = line.strip()
        if q:
            queries.append(q)

    return queries


def is_valid_query(q: str) -> bool:
    q_low = q.lower().strip()

    # blok frasa terlarang yang menunjukkan "bocor meta"
    if any(b in q_low for b in BANNED_PHRASES):
        return False

    # panjang kata
    n_words = len(q_low.split())
    if n_words < 4 or n_words > 25:
        return False

    return True



def generate_queries_from_tafsir(tafsir_text: str):
    prompt = build_prompt_for_tafsir(tafsir_text)

    resp = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {
                "role": "system",
                "content": (
                    "Kamu hanya membuat kalimat pencarian dalam Bahasa Indonesia "
                    "sesuai instruksi user."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        max_tokens=512,
        temperature=0.7,
        top_p=0.9,
    )

    text = resp.choices[0].message.content
    raw_queries = parse_queries_from_llm_output(text)

    # bersihkan & dedup lokal
    clean = []
    seen = set()
    for q in raw_queries:
        q_clean = str(q).strip()
        if not q_clean:
            continue
        key = q_clean.lower()
        if key in seen:
            continue
        seen.add(key)
        clean.append(q_clean)

    usage = {
        "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
        "completion_tokens": getattr(resp.usage, "completion_tokens", None),
        "total_tokens": getattr(resp.usage, "total_tokens", None),
    }

    return clean, usage


In [18]:
contoh_tafsir = df_tafsir.iloc[2][COL_TAFSIR]
print("Potongan tafsir:\n", contoh_tafsir[:400], "...\n")

qs, usage = generate_queries_from_tafsir(contoh_tafsir)
print("Queries:")
for i, q in enumerate(qs, 1):
    print(f"{i}.", q)
print("Usage:", usage)


Potongan tafsir:
 Dialah Yang Maha Pengasih, Pemilik dan sumber sifat kasih, Yang menganugerahkan segala macam karunia, baik besar maupun kecil, kepada seluruh makhluk, Maha Penyayang Yang selalu tiada henti memberi kasih dan kebaikan kepada orang-orang yang beriman. ...

Queries:
1. Apa perbedaan antara sifat Ar-Rahman dan Ar-Rahim dalam Al-Qur'an?
2. ayat tentang Allah Yang Maha Pengasih dan Maha Penyayang
3. Mengapa Allah memberikan karunia kepada semua makhluk, bukan hanya orang beriman?
4. dalil bahwa Allah sumber segala kasih sayang dan kebaikan
5. Bagaimana cara mensyukuri nikmat Allah yang besar maupun kecil?
6. Apakah kasih sayang Allah kepada orang beriman berbeda dengan makhluk lainnya?
Usage: {'prompt_tokens': 625, 'completion_tokens': 113, 'total_tokens': 738}


In [21]:
PART = 2          # ganti 1,2,3,4,5,6 sesuai Colab yang sedang kamu jalankan
NUM_PARTS = 6     # sekarang kita pakai 6 Colab

total_rows = len(df_tafsir)
rows_per_part = total_rows // NUM_PARTS
sisa = total_rows % NUM_PARTS

if PART <= sisa:
    start_idx = (PART - 1) * (rows_per_part + 1)
    end_idx = start_idx + (rows_per_part + 1)
else:
    start_idx = sisa * (rows_per_part + 1) + (PART - sisa - 1) * rows_per_part
    end_idx = start_idx + rows_per_part

print(f"TOTAL baris : {total_rows}")
print(f"PART {PART}/{NUM_PARTS} → rows {start_idx} : {end_idx}")

df_part = df_tafsir.iloc[start_idx:end_idx].reset_index(drop=True)
print("Shape df_part:", df_part.shape)


TOTAL baris : 6236
PART 2/6 → rows 1040 : 2080
Shape df_part: (1040, 7)


In [22]:
OUT_DIR = Path("/content/drive/MyDrive/fp_quran/queries_v2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_path = OUT_DIR / (f"queries_part{PART}.csv" if 'PART' in globals() else "queries_all.csv")
print("Output file:", out_path)

# resume jika file sudah ada
if out_path.exists():
    df_existing = pd.read_csv(out_path)
    processed_ids = set(df_existing["seg_id"].unique())
    all_rows = df_existing.to_dict("records")
    processed_ayat_count = len(processed_ids)
    print("File sudah ada, baris:", len(df_existing))
else:
    processed_ids = set()
    all_rows = []
    processed_ayat_count = 0
    print("Belum ada file, mulai baru.")

MAX_QUERY_PER_SEGMENT = 8
SLEEP_SEC = 0.5

df_target = df_part if 'df_part' in globals() else df_tafsir
total_ayat = len(df_target)
print("Total ayat yang akan diproses di run ini:", total_ayat)

for _, row in tqdm(df_target.iterrows(), total=total_ayat):
    seg_id = row[COL_ID]

    if seg_id in processed_ids:
        continue

    tafsir_text = str(row[COL_TAFSIR]).strip()
    if not tafsir_text:
        continue

    surah     = row[COL_SURAH]
    ayah      = row[COL_AYAH]
    tafsir_id = row[COL_TAFSIR_ID]

    try:
        queries, usage = generate_queries_from_tafsir(tafsir_text)
    except Exception as e:
        print(f"Error pada ID {seg_id} (Surah {surah}, Ayat {ayah}): {e}")
        time.sleep(5)
        continue

    # filter kualitas
    queries = [q for q in queries if is_valid_query(q)]

    # batasi jumlah per ayat
    if len(queries) > MAX_QUERY_PER_SEGMENT:
        queries = queries[:MAX_QUERY_PER_SEGMENT]

    if not queries:
        continue

    for i, q in enumerate(queries, start=1):
        all_rows.append({
            "seg_id": seg_id,
            "surah": surah,
            "ayah": ayah,
            "tafsir_id": tafsir_id,
            "query_index": i,
            "query_text": q,
            "prompt_tokens": usage["prompt_tokens"],
            "completion_tokens": usage["completion_tokens"],
            "total_tokens": usage["total_tokens"],
        })

    processed_ids.add(seg_id)
    processed_ayat_count += 1

    if processed_ayat_count % 10 == 0:
        df_tmp = pd.DataFrame(all_rows)
        # dedup global per text
        df_tmp["query_text_norm"] = df_tmp["query_text"].str.lower().str.strip()
        df_tmp = df_tmp.drop_duplicates(subset=["query_text_norm"]).drop(columns=["query_text_norm"])
        df_tmp.to_csv(out_path, index=False)
        print(f"[AUTO-SAVE] {processed_ayat_count} ayat → {len(df_tmp)} baris di {out_path}")

    time.sleep(SLEEP_SEC)

# save final
df_final = pd.DataFrame(all_rows)
df_final["query_text_norm"] = df_final["query_text"].str.lower().str.strip()
df_final = df_final.drop_duplicates(subset=["query_text_norm"]).drop(columns=["query_text_norm"])

df_final.to_csv(out_path, index=False)
print("Selesai. Ayat diproses:", processed_ayat_count)
print("Total baris tersimpan:", len(df_final))


Output file: /content/drive/MyDrive/fp_quran/queries_v2/queries_part2.csv
Belum ada file, mulai baru.
Total ayat yang akan diproses di run ini: 1040


  0%|          | 0/1040 [00:00<?, ?it/s]

[AUTO-SAVE] 10 ayat → 70 baris di /content/drive/MyDrive/fp_quran/queries_v2/queries_part2.csv
[AUTO-SAVE] 20 ayat → 139 baris di /content/drive/MyDrive/fp_quran/queries_v2/queries_part2.csv
[AUTO-SAVE] 30 ayat → 207 baris di /content/drive/MyDrive/fp_quran/queries_v2/queries_part2.csv
[AUTO-SAVE] 40 ayat → 275 baris di /content/drive/MyDrive/fp_quran/queries_v2/queries_part2.csv


KeyboardInterrupt: 